#Importing Libraries

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


#Training Data

In [9]:
corpus = [
    "deep learning models require large datasets",
    "machine learning algorithms find hidden patterns",
    "artificial intelligence is transforming industries",
    "data preprocessing is an important step",
    "feature engineering improves model performance",
    "supervised learning uses labeled data",
    "unsupervised learning finds structure in data",
    "reinforcement learning learns through rewards",
    "neural networks mimic the human brain",
    "training data impacts model accuracy",
    "overfitting reduces generalization ability",
    "regularization helps prevent overfitting",
    "gradient descent optimizes model parameters",
    "activation functions introduce non-linearity",
    "convolutional networks are used for images",
    "natural language processing handles text data",
    "transformers are powerful for language tasks",
    "big data drives modern machine learning",
    "model evaluation requires proper metrics",
    "cross validation improves reliability"
]

print("Number of unique sentences:", len(corpus))
for line in corpus:
    print("-", line)

Number of unique sentences: 20
- deep learning models require large datasets
- machine learning algorithms find hidden patterns
- artificial intelligence is transforming industries
- data preprocessing is an important step
- feature engineering improves model performance
- supervised learning uses labeled data
- unsupervised learning finds structure in data
- reinforcement learning learns through rewards
- neural networks mimic the human brain
- training data impacts model accuracy
- overfitting reduces generalization ability
- regularization helps prevent overfitting
- gradient descent optimizes model parameters
- activation functions introduce non-linearity
- convolutional networks are used for images
- natural language processing handles text data
- transformers are powerful for language tasks
- big data drives modern machine learning
- model evaluation requires proper metrics
- cross validation improves reliability


#Tokenization

In [10]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

sequences = tokenizer.texts_to_sequences(corpus)

print("Word Index:\n", tokenizer.word_index)
print("\nSequences:\n", sequences)

Word Index:
 {'learning': 1, 'data': 2, 'model': 3, 'machine': 4, 'is': 5, 'improves': 6, 'networks': 7, 'overfitting': 8, 'are': 9, 'for': 10, 'language': 11, 'deep': 12, 'models': 13, 'require': 14, 'large': 15, 'datasets': 16, 'algorithms': 17, 'find': 18, 'hidden': 19, 'patterns': 20, 'artificial': 21, 'intelligence': 22, 'transforming': 23, 'industries': 24, 'preprocessing': 25, 'an': 26, 'important': 27, 'step': 28, 'feature': 29, 'engineering': 30, 'performance': 31, 'supervised': 32, 'uses': 33, 'labeled': 34, 'unsupervised': 35, 'finds': 36, 'structure': 37, 'in': 38, 'reinforcement': 39, 'learns': 40, 'through': 41, 'rewards': 42, 'neural': 43, 'mimic': 44, 'the': 45, 'human': 46, 'brain': 47, 'training': 48, 'impacts': 49, 'accuracy': 50, 'reduces': 51, 'generalization': 52, 'ability': 53, 'regularization': 54, 'helps': 55, 'prevent': 56, 'gradient': 57, 'descent': 58, 'optimizes': 59, 'parameters': 60, 'activation': 61, 'functions': 62, 'introduce': 63, 'non': 64, 'linearit

In [15]:
word_index = tokenizer.word_index
total_words = len(word_index) + 1  # +1 because indexing starts from 1

print("Vocabulary size:", total_words)
print("\nWord index:")
print(word_index)

Vocabulary size: 86

Word index:
{'learning': 1, 'data': 2, 'model': 3, 'machine': 4, 'is': 5, 'improves': 6, 'networks': 7, 'overfitting': 8, 'are': 9, 'for': 10, 'language': 11, 'deep': 12, 'models': 13, 'require': 14, 'large': 15, 'datasets': 16, 'algorithms': 17, 'find': 18, 'hidden': 19, 'patterns': 20, 'artificial': 21, 'intelligence': 22, 'transforming': 23, 'industries': 24, 'preprocessing': 25, 'an': 26, 'important': 27, 'step': 28, 'feature': 29, 'engineering': 30, 'performance': 31, 'supervised': 32, 'uses': 33, 'labeled': 34, 'unsupervised': 35, 'finds': 36, 'structure': 37, 'in': 38, 'reinforcement': 39, 'learns': 40, 'through': 41, 'rewards': 42, 'neural': 43, 'mimic': 44, 'the': 45, 'human': 46, 'brain': 47, 'training': 48, 'impacts': 49, 'accuracy': 50, 'reduces': 51, 'generalization': 52, 'ability': 53, 'regularization': 54, 'helps': 55, 'prevent': 56, 'gradient': 57, 'descent': 58, 'optimizes': 59, 'parameters': 60, 'activation': 61, 'functions': 62, 'introduce': 63, 

#Create training sequences

In [11]:
input_sequences = []

for sentence in corpus:
    token_list = tokenizer.texts_to_sequences([sentence])[0]

    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

print("Total sequences:", len(input_sequences))
for seq in input_sequences[:10]:
    print(seq)

Total sequences: 86
[12, 1]
[12, 1, 13]
[12, 1, 13, 14]
[12, 1, 13, 14, 15]
[12, 1, 13, 14, 15, 16]
[4, 1]
[4, 1, 17]
[4, 1, 17, 18]
[4, 1, 17, 18, 19]
[4, 1, 17, 18, 19, 20]


#Pad the sequences

In [12]:
max_sequence_len = max([len(seq) for seq in input_sequences])

input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

print("Padded sequence example:\n", input_sequences[:5])

Padded sequence example:
 [[ 0  0  0  0 12  1]
 [ 0  0  0 12  1 13]
 [ 0  0 12  1 13 14]
 [ 0 12  1 13 14 15]
 [12  1 13 14 15 16]]


#Splitting input X and Output Y

In [16]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (86, 5)
Shape of y: (86, 86)


#Model building

In [17]:
model = Sequential([
    Embedding(input_dim=total_words, output_dim=10, input_length=max_sequence_len - 1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

#Model Training

In [34]:
history = model.fit(X, y, epochs=200, verbose=1)

Epoch 1/196
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 1.0000 - loss: 0.1264
Epoch 2/196
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 1.0000 - loss: 0.1248
Epoch 3/196
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 1.0000 - loss: 0.1234
Epoch 4/196
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 1.0000 - loss: 0.1216
Epoch 5/196
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 1.0000 - loss: 0.1205
Epoch 6/196
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 1.0000 - loss: 0.1196
Epoch 7/196
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 1.0000 - loss: 0.1189
Epoch 8/196
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 1.0000 - loss: 0.1182
Epoch 9/196
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 1.0000 - loss: 0.1174
Epoch 10/196
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 1.0000 - loss: 0.1155
Epoch 11/196
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 1.0000 - loss: 0.1146
Epoch 12/196
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 1.0000 - lo

#Custom Predictions

In [35]:
def generate_text(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')

        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        seed_text += " " + output_word
    return seed_text

##Sample Data to Predict

In [36]:
print(generate_text("training data impacts", 2))
print(generate_text("convolutional networks", 4))
print(generate_text("data preprocessing", 4))
print(generate_text("supervised learning", 3))
print(generate_text("transformers", 5))
print(generate_text("gradient descent optimizes", 2))

training data impacts model accuracy
convolutional networks are used for images
data preprocessing is an important step
supervised learning uses labeled data
transformers are powerful for language tasks
gradient descent optimizes model parameters
